In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import LabelEncoder

# Leave-one-individual-out (LOIO) validation -- Random Forest

Focused robustness check requested during paper revision: instead of the 5-fold
group-stratified CV used in `05a_feature_classification_ml.ipynb`, hold out one
*individual's* entire record at a time and test on it, to see whether
performance holds up on an individual the model has never seen.

**Scope** (see `figs1_individual_validation_data_variation.png`):

- **Datasets**: Burst_1/uncorrected (the raw-data baseline) and
  Burst_2/uncorrected (the final classifier's burst length).
  Bursts 2-4 are the *same* underlying recordings as Burst_1, split into
  progressively shorter child windows, so a given burst can never be pooled
  with another burst -- each is processed as its own separate LOIO run, tagged
  with `burst`/`correction_type` columns and concatenated afterwards for
  comparison, not merged into one training pool.
- **Target**: behaviour only (the activity target is not used).
- Random Forest only here. TabPFN and HydraMultiROCKET are run separately in
  `08b_individual_validation_features_tabpfn.ipynb` and
  `08c_individual_validation_timeseries_dl.ipynb`, reusing the exact same fold
  assignments built in this notebook.
- An individual is only used as a held-out *test* fold in a given burst if
  they have **>=30** total labelled samples *in that burst* -- otherwise their
  data is always included in training but they are never singled out as a
  test fold. This threshold is applied **independently per burst**: Burst_2
  has ~2x as many rows per individual as Burst_1 (same recordings, finer
  windowing), so its eligible set is larger (17/20 individuals vs. 13/20 for
  Burst_1) -- each burst uses all the data it has rather than being
  artificially restricted to Burst_1's smaller eligible set.
- **The behaviour model trains on the full label space -- no classes are
  dropped from training.** It sees all 8 behaviour classes, exactly like the
  main CV models in `05a_feature_classification_ml.ipynb`, so LOIO and the
  main CV are training-equivalent and genuinely comparable. Running,
  Self-scratching, and Sleeping are instead excluded only when computing
  pooled metrics (see the pooling cell): only 4, 7, and 6 of the 20
  individuals contribute examples of these classes in Burst_1 (8, 13, and 6
  in Burst_2), so most held-out folds have zero test examples of them anyway,
  and scoring on them would mostly measure whether a class was absent from
  that particular fold's test set rather than genuine model performance.
  Restricting evaluation (not training) to the 5 well-populated classes keeps
  a fair, low-noise comparison metric without changing what the model
  actually learned.

This notebook builds one fold lookup per burst and runs Random Forest across
both datasets. `08b`/`08c` load the fold lookup CSVs produced here rather
than recomputing them, so all three models are compared on identical
per-burst held-out sets.

## Diagnostics: per-individual and per-class sample counts, per burst

In [ ]:
fold_lookup_full = pd.read_csv("../data/temp/indices_cv_fold_lookup.csv")

BURSTS = ["burst_1", "burst_2"]

per_burst_diagnostics = {}
for burst in BURSTS:
    b = fold_lookup_full[fold_lookup_full["burst"] == burst].copy()
    individual_totals = b["Ind_ID"].value_counts().sort_values(ascending=False)
    individual_behaviour_counts = pd.crosstab(b["Ind_ID"], b["behaviour"])
    behaviour_summary = pd.DataFrame(
        {
            "total": individual_behaviour_counts.sum(axis=0),
            "n_individuals_ge1": (individual_behaviour_counts >= 1).sum(axis=0),
            "n_individuals_ge5": (individual_behaviour_counts >= 5).sum(axis=0),
            "n_individuals_ge10": (individual_behaviour_counts >= 10).sum(axis=0),
        }
    )
    per_burst_diagnostics[burst] = {
        "df": b,
        "individual_totals": individual_totals,
        "individual_behaviour_counts": individual_behaviour_counts,
        "behaviour_summary": behaviour_summary,
    }

    print(f"=== {burst} ===")
    print(f"Total rows: {len(b)}, unique individuals: {b['Ind_ID'].nunique()}")
    print("\nPer-individual total sample counts:")
    print(individual_totals)
    print("\nPer-behaviour spread across individuals:")
    print(behaviour_summary)
    print()

## Apply thresholds: eligible individuals (per burst) and retained behaviour classes (fixed across bursts)

In [ ]:
MIN_SAMPLES_FOR_LOIO_FOLD = 30
# Classes excluded from *pooled metrics only* (not from training -- see the
# pooling cell and the notebook introduction for why). Fixed across every
# burst/correction type we test, so results stay comparable across datasets --
# chosen from Burst_1's diagnostics. Burst_2's own numbers alone would
# arguably support keeping Self-scratching in (more individuals/counts at its
# finer child-window granularity), but changing which classes get evaluated
# per burst would reintroduce the same "different label space" confound.
DROPPED_MULTICLASS_CLASSES = ["Running", "Self-scratching", "Sleeping"]
TRAIN_ONLY_SENTINEL = "train_only"

per_burst_eligibility = {}
for burst in BURSTS:
    diag = per_burst_diagnostics[burst]
    individual_totals = diag["individual_totals"]
    individual_behaviour_counts = diag["individual_behaviour_counts"]

    eligible = sorted(
        individual_totals[individual_totals >= MIN_SAMPLES_FOR_LOIO_FOLD].index.tolist()
    )
    excluded = sorted(
        individual_totals[individual_totals < MIN_SAMPLES_FOR_LOIO_FOLD].index.tolist()
    )
    per_burst_eligibility[burst] = {"eligible": eligible, "excluded": excluded}

    print(f"=== {burst} ===")
    print(f"Eligible LOIO fold individuals ({len(eligible)}): {eligible}")
    print(f"Excluded, kept in training only ({len(excluded)}): {excluded}")

    retained_classes = [
        c
        for c in individual_behaviour_counts.columns
        if c not in DROPPED_MULTICLASS_CLASSES
    ]
    coverage = individual_behaviour_counts.loc[eligible, retained_classes]
    print("Per-class fold coverage across eligible individuals:")
    print(
        pd.DataFrame(
            {
                "n_folds_with_data": (coverage >= 1).sum(axis=0),
                "n_eligible_folds": len(eligible),
            }
        )
    )
    print()

# The set of classes pooled multiclass metrics are restricted to -- the
# complement of DROPPED_MULTICLASS_CLASSES within whatever classes actually
# appear (unioned across bursts, in case one burst had an extra rare class
# the other didn't).
all_multiclass_classes = sorted(
    set().union(
        *[
            per_burst_diagnostics[b]["individual_behaviour_counts"].columns
            for b in BURSTS
        ]
    )
)
RETAINED_MULTICLASS_CLASSES = [
    c for c in all_multiclass_classes if c not in DROPPED_MULTICLASS_CLASSES
]
print(
    f"Retained multiclass classes for metrics ({len(RETAINED_MULTICLASS_CLASSES)}): "
    f"{RETAINED_MULTICLASS_CLASSES}"
)

## Build and save one LOIO fold lookup per burst

Fold lookups are per-burst, not per-correction-type: correction only changes
the feature values, not which samples/individuals/labels exist, so one fold
lookup per burst would be reused across every correction type run against it.

In [ ]:
loio_lookup_paths = {}
for burst in BURSTS:
    b = per_burst_diagnostics[burst]["df"]
    eligible = per_burst_eligibility[burst]["eligible"]

    loio_lookup = b.copy()
    loio_lookup["fold"] = loio_lookup["Ind_ID"].where(
        loio_lookup["Ind_ID"].isin(eligible), TRAIN_ONLY_SENTINEL
    )

    path = f"../data/temp/indices_loio_fold_lookup_{burst}.csv"
    loio_lookup.to_csv(path, index=False)
    loio_lookup_paths[burst] = path
    print(f"Saved {len(loio_lookup)} rows -> {path}")

## Random Forest LOIO training/evaluation function

In [ ]:
def features_ml_models(
    X_train,
    X_test,
    y_train,
    y_test,
    class_names,
    use_class_weight=True,
):
    """Train and evaluate Random Forest on a pre-split train/test pair (one LOIO fold).

    Mirrors features_ml_models in 05a_feature_classification_ml.ipynb, trimmed
    to Random Forest only (this analysis focuses on RF, TabPFN, and
    HydraMultiROCKET -- see 08b/08c for the other two).
    """
    cw = "balanced" if use_class_weight else None
    model = RandomForestClassifier(n_estimators=100, class_weight=cw, random_state=42)
    n_classes = len(class_names)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        t0 = time.time()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        proba = model.predict_proba(X_test)
        try:
            roc = roc_auc_score(y_test, proba, multi_class="ovr", average="weighted")
        except Exception:
            roc = np.nan

        metrics_results = [
            {
                "model": "RF",
                "roc_auc": roc,
                "accuracy": model.score(X_test, y_test),
                "balanced_acc": balanced_accuracy_score(y_test, y_pred),
                "f1_score": f1_score(
                    y_test, y_pred, average="weighted", zero_division=0
                ),
                "precision": precision_score(
                    y_test, y_pred, average="weighted", zero_division=0
                ),
                "recall": recall_score(
                    y_test, y_pred, average="weighted", zero_division=0
                ),
                "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
                "precision_macro": precision_score(
                    y_test, y_pred, average="macro", zero_division=0
                ),
                "recall_macro": recall_score(
                    y_test, y_pred, average="macro", zero_division=0
                ),
                "time": time.time() - t0,
            }
        ]

        cm = confusion_matrix(y_test, y_pred, labels=np.arange(n_classes))
        confusion_data = [
            {
                "model": "RF",
                "actual_label": class_names[i],
                "predicted_label": class_names[j],
                "count": cm[i][j],
            }
            for i in range(n_classes)
            for j in range(n_classes)
        ]

    return pd.DataFrame(metrics_results), pd.DataFrame(confusion_data)

## LOIO orchestration for the behaviour target

In [ ]:
def process_parquet_file(
    file_path, fold_lookup, eligible_individuals, use_class_weight=True
):
    """Run LOIO cross-validation for the behaviour target on one file.

    The model trains on the full label space -- no classes are dropped here.
    It sees all 8 behaviour classes, exactly like the main CV models in
    05a_feature_classification_ml.ipynb, so LOIO and the main CV are
    training-equivalent. Sparse classes are excluded only when computing
    pooled metrics afterward (see the pooling cell), not from training.

    Args:
        file_path (str): Path to the parquet file
        fold_lookup (pd.DataFrame): LOIO fold lookup with columns burst,
            Ind_ID, new_burst, behaviour, activity, fold -- fold is the
            held-out individual's name for eligible individuals, or the
            train_only sentinel for individuals never used as a test fold
        eligible_individuals (list): Individual names to iterate as held-out folds
        use_class_weight (bool): Whether to use class_weight="balanced" for RF

    Returns:
        dict: multiclass_metrics and multiclass_conf DataFrames
    """
    burst = os.path.basename(os.path.dirname(file_path))
    correction_type = os.path.basename(file_path).split("_")[-1].split(".")[0]
    print(f"Processing {correction_type} for {burst}")

    features = pd.read_parquet(file_path)
    m = fold_lookup[["Ind_ID", "new_burst", "behaviour", "activity", "fold"]]
    df = features.merge(m, on=["Ind_ID", "new_burst"], how="inner")

    X_full = df.drop(
        ["Ind_ID", "new_burst", "behaviour", "activity", "fold"],
        axis=1,
        errors="ignore",
    )

    enc = LabelEncoder()
    y_enc = enc.fit_transform(df["behaviour"])
    class_names = list(enc.classes_)

    metrics, confs = [], []
    for held_out in eligible_individuals:
        tr = (df["fold"] != held_out).to_numpy()
        te = (df["fold"] == held_out).to_numpy()
        if te.sum() == 0:
            continue
        met, conf = features_ml_models(
            X_full[tr],
            X_full[te],
            y_enc[tr],
            y_enc[te],
            class_names,
            use_class_weight=use_class_weight,
        )
        for d in (met, conf):
            d["correction_type"] = correction_type
            d["burst"] = burst
            d["target"] = "Behaviour"
            d["held_out_individual"] = held_out
            d["n_test_samples"] = int(te.sum())
        metrics.append(met)
        confs.append(conf)

    metric_columns = [
        "burst",
        "correction_type",
        "target",
        "held_out_individual",
        "n_test_samples",
        "model",
        "roc_auc",
        "accuracy",
        "balanced_acc",
        "f1_score",
        "precision",
        "recall",
        "f1_macro",
        "precision_macro",
        "recall_macro",
        "time",
    ]
    conf_columns = [
        "burst",
        "correction_type",
        "target",
        "held_out_individual",
        "n_test_samples",
        "model",
        "actual_label",
        "predicted_label",
        "count",
    ]
    fin = lambda lst, cols: pd.concat(lst)[cols] if lst else pd.DataFrame(columns=cols)
    return {
        "multiclass_metrics": fin(metrics, metric_columns),
        "multiclass_conf": fin(confs, conf_columns),
    }

## Run LOIO across datasets

In [ ]:
DATASETS = [
    ("burst_1", "uncorrected"),
    ("burst_2", "uncorrected"),
]

all_results = []
for burst, correction_type in DATASETS:
    fold_lookup = pd.read_csv(loio_lookup_paths[burst])
    eligible = per_burst_eligibility[burst]["eligible"]
    pq_file_path = (
        f"../data/raw/features/{burst.capitalize()}/"
        f"annotated_features_{burst}_{correction_type}.parquet"
    )
    result = process_parquet_file(pq_file_path, fold_lookup, eligible)
    all_results.append(result)

multiclass_metrics = pd.concat(
    [r["multiclass_metrics"] for r in all_results], ignore_index=True
)
multiclass_conf = pd.concat(
    [r["multiclass_conf"] for r in all_results], ignore_index=True
)

## Pool results and inspect per-individual breakdown

In [ ]:
def pooled_metrics_from_confusion(conf_df, model_name, retained_classes=None):
    """Reconstruct pooled y_true/y_pred from a summed confusion matrix and
    compute overall metrics on it -- more meaningful than averaging
    per-individual metrics when many individuals have zero test examples of
    some classes (e.g. grooming for individuals who never groom).

    Args:
        retained_classes (list, optional): If given, metrics are computed
            only over rows whose TRUE label is one of these classes, keeping
            whatever the model actually predicted (even a class outside this
            set counts as a genuine miss). Training itself is never
            restricted -- see process_parquet_file -- this only controls
            which true-label rows count toward the reported metrics, so the
            model stays an 8-class classifier and only the comparison metric
            is restricted to the well-populated classes.
    """
    sub = conf_df[conf_df["model"] == model_name]
    if sub.empty or sub["count"].sum() == 0:
        return {"model": model_name, "n_pooled_test_samples": 0}
    y_true = np.repeat(sub["actual_label"].to_numpy(), sub["count"].to_numpy())
    y_pred = np.repeat(sub["predicted_label"].to_numpy(), sub["count"].to_numpy())

    if retained_classes is not None:
        mask = np.isin(y_true, retained_classes)
        y_true, y_pred = y_true[mask], y_pred[mask]
        labels_kwarg = {"labels": retained_classes}
    else:
        labels_kwarg = {}

    if len(y_true) == 0:
        return {"model": model_name, "n_pooled_test_samples": 0}

    # balanced_accuracy_score has no `labels` param, so when restricting to a
    # subset of classes we use macro recall instead -- mathematically the
    # same thing balanced accuracy is (mean per-class recall), and this
    # matches the convention already used elsewhere (recall_macro ==
    # balanced_acc throughout 05a-05d's own metrics).
    balanced_acc = (
        recall_score(y_true, y_pred, average="macro", zero_division=0, **labels_kwarg)
        if retained_classes is not None
        else balanced_accuracy_score(y_true, y_pred)
    )

    return {
        "model": model_name,
        "n_pooled_test_samples": len(y_true),
        "accuracy": float((y_true == y_pred).mean()),
        "balanced_acc": balanced_acc,
        "f1_score": f1_score(
            y_true, y_pred, average="weighted", zero_division=0, **labels_kwarg
        ),
        "precision": precision_score(
            y_true, y_pred, average="weighted", zero_division=0, **labels_kwarg
        ),
        "recall": recall_score(
            y_true, y_pred, average="weighted", zero_division=0, **labels_kwarg
        ),
        "f1_macro": f1_score(
            y_true, y_pred, average="macro", zero_division=0, **labels_kwarg
        ),
        "precision_macro": precision_score(
            y_true, y_pred, average="macro", zero_division=0, **labels_kwarg
        ),
        "recall_macro": recall_score(
            y_true, y_pred, average="macro", zero_division=0, **labels_kwarg
        ),
    }


def pooled_summary_per_dataset(conf_df, model_name, retained_classes=None):
    """Pooled metrics computed separately per (burst, correction_type), so
    datasets are never mixed into a single pool (Burst_2 is the same
    recordings as Burst_1 at finer windowing -- pooling across bursts would
    leak near-duplicate windows across train/test)."""
    rows = []
    for (burst, correction_type), grp in conf_df.groupby(["burst", "correction_type"]):
        m = pooled_metrics_from_confusion(
            grp, model_name, retained_classes=retained_classes
        )
        m["burst"] = burst
        m["correction_type"] = correction_type
        rows.append(m)
    cols = ["burst", "correction_type"] + [
        c for c in rows[0] if c not in ("burst", "correction_type")
    ]
    return pd.DataFrame(rows)[cols]


pooled_multiclass_summary = pooled_summary_per_dataset(
    multiclass_conf, "RF", retained_classes=RETAINED_MULTICLASS_CLASSES
)
print("Pooled multiclass LOIO metrics per dataset (restricted to retained classes):")
print(pooled_multiclass_summary)

# Per-individual breakdown, alongside their sample counts, for transparency --
# this is the honest complement to the pooled summary: it shows how much the
# pooled result is driven by well-sampled vs. sparse individuals. Note these
# per-individual accuracy/balanced_acc/f1_macro values come straight from
# features_ml_models and are NOT restricted to RETAINED_MULTICLASS_CLASSES
# (that restriction is only applied in the pooled summary above) -- they
# reflect genuine 8-class performance for that held-out individual.
per_individual_cols = [
    "burst",
    "correction_type",
    "held_out_individual",
    "n_test_samples",
    "model",
    "accuracy",
    "balanced_acc",
    "f1_macro",
]
per_individual_multiclass = multiclass_metrics[per_individual_cols].sort_values(
    ["burst", "correction_type", "n_test_samples"], ascending=[True, True, False]
)
print("\nPer-individual multiclass results (unrestricted, all 8 classes):")
print(per_individual_multiclass)

## Save outputs

In [ ]:
multiclass_metrics.to_csv(
    "../data/output/comparison_behaviour/behaviour_features_ml_loio_metrics.csv",
    index=False,
)
multiclass_conf.to_csv(
    "../data/output/comparison_behaviour/behaviour_features_ml_loio_confusion.csv",
    index=False,
)
pooled_multiclass_summary.to_csv(
    "../data/output/comparison_behaviour/behaviour_features_ml_loio_pooled_summary.csv",
    index=False,
)